# Wine Quality Classification - Model Training Notebook

This notebook demonstrates training and evaluating machine learning models for wine quality prediction.

## Models Trained:
1. Logistic Regression (baseline)
2. Random Forest (ensemble)
3. XGBoost/LightGBM (boosting)

## Workflow:
- Data loading and preprocessing
- Hyperparameter tuning using GridSearchCV/RandomizedSearchCV
- Model evaluation using multiple metrics
- Comparison and visualization

In [ ]:
# Import necessary libraries
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd()
sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.fase_2.models import ModelTrainer

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print("Libraries imported successfully!")

In [ ]:
# Initialize the ModelTrainer
print("Initializing ModelTrainer...")

trainer = ModelTrainer(
    train_path="data/processed/train.csv",
    output_dir="models",
    results_dir="results"
)

print("ModelTrainer initialized successfully!")

In [ ]:
# Load and preprocess data
print("Loading data...")
trainer.load_and_preprocess_data(stratify=True)

# Display data info
print("\nX_train shape:", trainer.X_train.shape)
print("X_test shape:", trainer.X_test.shape)
print("\nTarget distribution in train set:")
print(trainer.y_train.value_counts().sort_index())

## Model 1: Logistic Regression (Baseline)

In [ ]:
# Train Logistic Regression with GridSearchCV
print("Training Logistic Regression...")

trainer.train_logistic_regression(cv_folds=5)

## Model 2: Random Forest

In [ ]:
# Train Random Forest with RandomizedSearchCV
print("\nTraining Random Forest...")

trainer.train_random_forest(cv_folds=5, n_iter=50)

## Model 3: XGBoost/LightGBM

In [ ]:
# Train XGBoost (using LightGBM as fallback) with RandomizedSearchCV
print("\nTraining XGBoost/LightGBM...")

trainer.train_xgboost(cv_folds=5, n_iter=50)

## Model Comparison

In [ ]:
# Compare all models
comparison = trainer.compare_models()

print("\nComparison DataFrame:")
display(comparison)

In [ ]:
# Visualize model comparison
plt.figure(figsize=(10, 6))
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
x = np.arange(len(metrics))
width = 0.25

for i, model in enumerate(comparison['Model']):
    plt.bar(x + i*width, comparison.loc[comparison['Model'] == model, metrics].values[0], width, label=model)

plt.xlabel('Metrics')
plt.ylabel('Score')
plt.title('Model Performance Comparison')
plt.xticks(x + width, metrics)
plt.legend()
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

## Save Models and Results

In [ ]:
# Save trained models
trainer.save_models()

# Save evaluation results
trainer.save_results()

print("\nModels and results saved successfully!")

## View Saved Results

In [ ]:
# Load and display saved results
import json

results_path = Path('results/model_performance.json')

with open(results_path, 'r') as f:
    results = json.load(f)

print("Best Parameters:")
print(json.dumps(results['best_params'], indent=2))

print("\nModel Performance:")
for model, metrics in results['model_metrics'].items():
    print(f"\n{model}:")
    for key, value in metrics.items():
        if key != 'classification_report' and key != 'confusion_matrix':
            print(f"  {key}: {value:.4f}")

In [ ]:
# Show classification report for best model
best_model_name = comparison['Model'].iloc[comparison['F1-Score'].idxmax()]
print(f"Best Model: {best_model_name}")
print("\nClassification Report:")
print(json.dumps(results['model_metrics'][best_model_name]['classification_report'], indent=2))

## Conclusion

In this notebook, we:
1. Trained three classification models: Logistic Regression, Random Forest, and XGBoost/LightGBM
2. Performed hyperparameter tuning using GridSearchCV and RandomizedSearchCV
3. Evaluated models using accuracy, precision, recall, and F1-score
4. Compared model performance and identified the best model
5. Saved trained models and evaluation results to disk